# 6-LoRA

对应 `model/model_lora.py` 与 `trainer/train_lora.py`。在方阵 Linear（`in_features == out_features`）上挂 `xW + BAx`，只训练 A/B。

![LoRA](./images/lora.png)

训练完可用 `scripts/convert_model.py` 的 `convert_merge_base_lora` 把 LoRA 合并回基模。


In [ ]:
import os, sys, math, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.append(os.path.abspath(".."))
import torch
from torch import optim
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from dataset.lm_dataset import PretrainDataset, SFTDataset, DPODataset, RLAIFDataset, AgentRLDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("../model")
print("device:", device, "vocab:", tokenizer.vocab_size)
def tiny_model(use_moe=False):
    cfg = MiniMindConfig(hidden_size=64, num_hidden_layers=1, use_moe=use_moe)
    return MiniMindForCausalLM(cfg).to(device), cfg

from model.model_lora import LoRA, apply_lora, save_lora


In [ ]:
model, cfg = tiny_model()
apply_lora(model, rank=4)
lora_params = [p for n, p in model.named_parameters() if "lora" in n]
print("lora tensors:", len(lora_params), "numel:", sum(p.numel() for p in lora_params))
for n, p in model.named_parameters():
    if "lora" not in n:
        p.requires_grad = False


数据格式与 SFT 相同，只是通常用垂域小数据（医疗、自我认知等）。


In [ ]:
ds = SFTDataset("./toydata/lora_data.jsonl", tokenizer, max_length=96)
loader = DataLoader(ds, batch_size=2)
optimizer = optim.AdamW(lora_params, lr=1e-4)
model.train()
for step, (input_ids, labels) in enumerate(loader, start=1):
    input_ids, labels = input_ids.to(device), labels.to(device)
    out = model(input_ids, labels=labels)
    loss = out.loss + out.aux_loss
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(f"step {step} loss={float(loss):.4f}")
os.makedirs("./out", exist_ok=True)
save_lora(model, "./out/lora_identity_64.pth")
print("saved ./out/lora_identity_64.pth")


完整训练：`cd trainer && python train_lora.py`。测试：`python eval_llm.py --weight full_sft --lora_weight lora_medical`。
